In [ ]:
import pandas as pd
import numpy as np

In [ ]:
X_train = pd.read_csv("../data/X_train_data.csv")
X_test = pd.read_csv("../data/X_test_data.csv")

In [ ]:
X_train.columns

Index(['Unnamed: 0', 'Name', 'Genres', 'Type', 'Episodes', 'Aired',
       'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
       'Ranked', 'Popularity', 'Members', 'Favorites', 'Watching', 'Completed',
       'On-Hold', 'Dropped', 'Plan to Watch'],
      dtype='object')

## Data Cleaning

Some columns will be dropped, and some will be transformed

### Columns that will be largely unchanged (missing values will be filled in):
 - Ranked
 - Popularity
 - Members
 - Favorites
 - Watching
 - Completed
 - On-hold
 - Dropped
 - Plan to watch

### Columns to be transformed:
 - Genres (One hot encoding)
 - Type (One hot encoding)
 - Source (One hot encoding)
 - Rating (One hot encoding)
 - Aired (extracting the year)
 - Duration (extracting the seconds)

### Columns that will be dropped:
 - Unnamed: 0
 - MAL_ID
 - English name (there is already a name variable)
 - Japanese name (there is already a name variable)
 - Premiered 
 - Name
 - Producers
 - Licensors
 - Studios
 - Score-10
 - Score-9
 - Score-8
 - Score-7
 - Score-6
 - Score-5
 - Score-4
 - Score-3
 - Score-2
 - Score-1

### Columns that have improper data types or missing values that will be altered:
 - Episodes
 - 

Additionally, some rows will be filled in with averages to ensure there are no missing values

### One hot encoding Columns

In [ ]:
cols = ["Genres", "Type", "Source", "Rating"]
for col in cols:
    dummies = pd.get_dummies(X_train[col], prefix=col)
    X_train = X_train.join(dummies)
    X_train.drop(columns=[col], inplace=True)


### Air Date Cleaning (Extracting the air year)

In [ ]:
#Dealing with the aired column now
X_train["Aired"].unique()

array(['Oct 21, 1994', 'Oct 11, 1999', 'Sep 13, 2003', ...,
       'Oct 5, 2010 to Dec 28, 2010', 'Apr 14, 2006 to Jun 23, 2006',
       'Apr 20, 2019'], shape=(10013,), dtype=object)

In [ ]:
X_train['Year'] = X_train['Aired'].str.extract(r'(\d{4})').astype('float').astype('Int64')

In [ ]:
X_train.drop("Aired", axis = 1, inplace = True)

### Cleaning the Duration column (length in seconds)

In [ ]:
#dealing with the duration column
X_train.Duration.head()

0    28 min. per ep.
1            54 min.
2            20 min.
3    24 min. per ep.
4    30 min. per ep.
Name: Duration, dtype: object

In [ ]:
import re

def extract_seconds(s):
    s = str(s).lower()

    hr_match  = re.search(r'(\d+)\s*hr', s)
    min_match = re.search(r'(\d+)\s*min', s)
    sec_match = re.search(r'(\d+)\s*sec', s)

    hours = int(hr_match.group(1)) if hr_match else 0
    minutes = int(min_match.group(1)) if min_match else 0
    seconds = int(sec_match.group(1)) if sec_match else 0

    total_seconds = hours*3600 + minutes*60 + seconds

    return total_seconds if total_seconds > 0 else None

In [ ]:
X_train["Seconds"] = X_train["Duration"].apply(extract_seconds)

In [ ]:
X_train.drop("Duration", axis =1, inplace = True)
print("Dropped Duration")

Dropped Duration


In [ ]:
X_train.drop("Unnamed: 0", axis=1, inplace = True)


### Dealing with missing values or wrong data types

In [ ]:
X_train.columns

Index(['Name', 'Episodes', 'Producers', 'Licensors', 'Studios', 'Ranked',
       'Popularity', 'Members', 'Favorites', 'Watching',
       ...
       'Source_Web manga', 'Rating_G - All Ages', 'Rating_PG - Children',
       'Rating_PG-13 - Teens 13 or older',
       'Rating_R - 17+ (violence & profanity)', 'Rating_R+ - Mild Nudity',
       'Rating_Rx - Hentai', 'Rating_Unknown', 'Year', 'Seconds'],
      dtype='object', length=4425)

In [ ]:
for col in ["Ranked", 'Popularity', 'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped', 'Plan to Watch']:
    X_train[col] = X_train[col].replace("Unknown", -1)
    X_train[col] = X_train[col].astype(float)
    median_value = X_train[col].median()
    X_train[col] = X_train[col].replace(-1, median_value )

In [ ]:
X_train.dtypes

Name                        object
Episodes                    object
Producers                   object
Licensors                   object
Studios                     object
                            ...   
Rating_R+ - Mild Nudity       bool
Rating_Rx - Hentai            bool
Rating_Unknown                bool
Year                         Int64
Seconds                    float64
Length: 4425, dtype: object

In [ ]:
# # Identify numerical and categorical columns
# num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
# cat_cols = X_train.select_dtypes(include=['object']).columns

# # Fill missing numerical values with median
# X_train[num_cols] = X_train[num_cols].fillna(0)

# # Fill missing categorical values with "Unknown"
# X_train[cat_cols] = X_train[cat_cols].fillna("Unknown")

# X_train.isnull().sum()

In [ ]:
X_train.to_csv("Cleaned_X_train.csv")